In [1]:
import pandas as pd
import json

from pymongo import MongoClient

In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["censo_locales_db"]
collection = db["locales"]

a. El Total de locales y terrazas por distrito y barrio: construye una consulta que
permita obtener el total de locales y terrazas agrupados por cada distrito y
barrio.

• Pista: identifica en tu modelo los campos correspondientes a distrito y
barrio. Usa un pipeline de agregación y aplica $group para agrupar los
documentos por estas categorías. Utiliza $sum para contar el total de
locales y terrazas en cada grupo.


In [3]:
pipeline_a = [
    {
        "$group": {
            # Clave de agrupación:
            # Se agrupa por distrito y barrio usando campos anidados del documento "local"
            "_id": {
                "distrito": "$local.desc_distrito_local",
                "barrio": "$local.desc_barrio_local"
            },
            
            
            "total_locales": {"$sum": 1}, # Conteo total de locales en cada grupo
            
            # Suma total de terrazas:
            # Se calcula el tamaño del arreglo "terrazas" por documento y luego se suma
            "total_terrazas": {"$sum": {"$size": "$terrazas"}}
        }
    },
    {
        "$sort": {
            "_id.distrito": 1,
            "_id.barrio": 1
        }
    }
]

resultado_a = list(collection.aggregate(pipeline_a))

for doc in resultado_a[:5]: # truncar uot
    print(doc)

{'_id': {'distrito': 'ARGANZUELA          ', 'barrio': 'ACACIAS             '}, 'total_locales': 1236, 'total_terrazas': 97}
{'_id': {'distrito': 'ARGANZUELA          ', 'barrio': 'ATOCHA              '}, 'total_locales': 246, 'total_terrazas': 6}
{'_id': {'distrito': 'ARGANZUELA          ', 'barrio': 'CHOPERA             '}, 'total_locales': 878, 'total_terrazas': 50}
{'_id': {'distrito': 'ARGANZUELA          ', 'barrio': 'DELICIAS            '}, 'total_locales': 862, 'total_terrazas': 71}
{'_id': {'distrito': 'ARGANZUELA          ', 'barrio': 'IMPERIAL            '}, 'total_locales': 826, 'total_terrazas': 61}


b. Tipos de licencias y cantidad de licencias por cada tipo: crea una consulta
para contar cuántas licencias hay de cada tipo en los datos.

• Pista: encuentra el campo que indica el tipo de licencia. Utiliza $group
para agrupar por este campo y $count o $sum para calcular el número
de licencias en cada grupo. Si encuentras valores nulos o
inconsistentes, considera cómo manejarlos.


In [5]:
pipeline_b = [
    {"$unwind": "$licencias"},
    {
        "$group": {
            "_id": "$licencias.desc_tipo_licencia",
            "total": {"$sum": 1} # $count
        }
    },
    {"$sort": {"total": -1}}
]

resultado_b = list(collection.aggregate(pipeline_b))
for doc in resultado_b[:5]: # truncar uot
    print(doc)

{'_id': 'Transmisión de licencia Urbanística', 'total': 57028}
{'_id': 'Declaración Responsable', 'total': 51249}
{'_id': 'Licencia Urbanística', 'total': 30783}
{'_id': 'Licencia recogida en el trabajo de campo', 'total': 5956}
{'_id': 'Licencia de Funcionamiento', 'total': 5813}


c. Listado de locales y terrazas con licencias “En trámite”: diseña una consulta
que filtre y devuelva un listado detallado de locales y terrazas cuyo estado de
licencia sea “En trámite”.

• Pista: localiza el campo que representa el estado de la licencia. Usa el
operador $match para filtrar los documentos donde este campo sea
igual a “En trámite”. Piensa también en cómo manejar mayúsculas,
minúsculas u otros formatos en el texto.

In [6]:
pipeline_c = [
    {"$unwind": "$licencias"},
    {
        "$match": {
            "licencias.desc_tipo_situacion_licencia": {
                "$regex": "En tramitación",
                "$options": "i"
            }
        }
    },
    {
        "$project": {
            "id_local": 1,
            "licencias.desc_tipo_situacion_licencia": 1,
            "local.desc_distrito_local": 1,
            "local.desc_barrio_local": 1,
        }
    }
]

resultado_c = list(collection.aggregate(pipeline_c))

for doc in resultado_c[:5]: # truncar uot
    print(doc)

{'_id': 20000596, 'local': {'desc_distrito_local': 'ARGANZUELA          ', 'desc_barrio_local': 'CHOPERA             '}, 'licencias': {'desc_tipo_situacion_licencia': 'En tramitación'}}
{'_id': 20000709, 'local': {'desc_distrito_local': 'ARGANZUELA          ', 'desc_barrio_local': 'ACACIAS             '}, 'licencias': {'desc_tipo_situacion_licencia': 'En tramitación'}}
{'_id': 20000709, 'local': {'desc_distrito_local': 'ARGANZUELA          ', 'desc_barrio_local': 'ACACIAS             '}, 'licencias': {'desc_tipo_situacion_licencia': 'En tramitación'}}
{'_id': 20000729, 'local': {'desc_distrito_local': 'ARGANZUELA          ', 'desc_barrio_local': 'DELICIAS            '}, 'licencias': {'desc_tipo_situacion_licencia': 'En tramitación'}}
{'_id': 20000729, 'local': {'desc_distrito_local': 'ARGANZUELA          ', 'desc_barrio_local': 'DELICIAS            '}, 'licencias': {'desc_tipo_situacion_licencia': 'En tramitación'}}


d. Consulta por sección, división y epígrafe de la actividad comercial: crea una
consulta para clasificar locales y terrazas según los campos sección, división y
epígrafe.

• Pista: busca estos tres campos en tu modelo y utiliza $and o construye
un filtro que combine las condiciones necesarias para devolver los
resultados clasificados. Considera cómo manejar casos donde uno de
estos campos pueda estar vacío.

In [7]:
pipeline_d = [
    {"$unwind": "$actividadeconomica"},
    {
        "$group": {
            "_id": {
                "id_local" : "$local.id_local",
                "id_seccion": "$actividadeconomica.id_seccion",
                "desc_seccion": "$actividadeconomica.desc_seccion",
                "id_division": "$actividadeconomica.id_division",
                "desc_division": "$actividadeconomica.desc_division",
                "id_epigrafe": "$actividadeconomica.id_epigrafe",
                "desc_epigrafe": "$actividadeconomica.desc_epigrafe"
                # ni idea cual dejar, suponiendo que tenga relacion y entre sufijos sean unicos la relacion
            },
            "total": {"$sum": 1}
        }
    },
    {"$sort": {"total": -1}}
]

resultado_d = list(collection.aggregate(pipeline_d))
for doc in resultado_d[:15]: # truncar uot
    print(doc)

{'_id': {'id_local': 280032650, 'id_seccion': 'Q', 'desc_seccion': 'ACTIVIDADES SANITARIAS Y DE SERVICIOS SOCIALES', 'id_division': '88', 'desc_division': 'ACTIVIDADES DE SERVICIOS SOCIALES SIN ALOJAMIENTO', 'id_epigrafe': '889002', 'desc_epigrafe': 'OTROS ACTIVIDADES DE SERVICIOS SOCIALES (LABORES DE ASESORAMIENTO Y ORIENTACION) SIN ALOJAMIENTO N.C.O.P.'}, 'total': 1}
{'_id': {'id_local': 280071694, 'id_seccion': 'S', 'desc_seccion': 'OTROS SERVICIOS', 'id_division': '96', 'desc_division': 'OTROS SERVICIOS PERSONALES', 'id_epigrafe': '960201', 'desc_epigrafe': 'SERVICIO DE PELUQUERIA'}, 'total': 1}
{'_id': {'id_local': 280061898, 'id_seccion': 'P', 'desc_seccion': 'EDUCACIÓN', 'id_division': '85', 'desc_division': 'EDUCACIÓN', 'id_epigrafe': '852001', 'desc_epigrafe': 'CENTRO DE INFANTIL Y PRIMARIA PUBLICO'}, 'total': 1}
{'_id': {'id_local': 285026926, 'id_seccion': '-1', 'desc_seccion': 'VALOR NULO EN ORIGEN', 'id_division': '-1', 'desc_division': 'VALOR NULO EN ORIGEN', 'id_epigrafe

e. Actividad económica más frecuente por barrio y distrito: diseña una consulta
que identifique cuál es la actividad económica predominante en cada barrio y
distrito.

• Pista: agrupa los datos por barrio y distrito usando $group. Luego,
dentro de cada grupo, cuenta la frecuencia de cada actividad
económica (campo correspondiente) y utiliza $sort o $max para
encontrar la más frecuente. Piensa en cómo combinar varias etapas
del pipeline para lograr esto. 

In [8]:
pipeline_e = [
    {"$unwind": "$actividadeconomica"},
    {
        "$group": {
            "_id": {
                "distrito": "$local.desc_distrito_local",
                "barrio": "$local.desc_barrio_local",
                "actividad": "$actividadeconomica.desc_seccion"
            },
            "total": {"$sum": 1}
        }
    },
    {"$sort": {"total": -1}},
    {
        "$group": {
            "_id": {
                "distrito": "$_id.distrito",
                "barrio": "$_id.barrio"
            },
            "actividad_mas_frecuente": {"$first": "$_id.actividad"},
            "total": {"$first": "$total"} 
        }
    },
    {
        "$project": {
            "_id": 0,
            "distrito": "$_id.distrito",
            "barrio": "$_id.barrio",
            "actividad_mas_frecuente": 1,
            "total": 1
        }
    }

]


resultado_e = list(collection.aggregate(pipeline_e))
for doc in resultado_e[:15]: # truncar uot
    print(doc)

{'actividad_mas_frecuente': 'COMERCIO AL POR MAYOR Y AL POR MENOR; REPARACIÓN DE VEHÍCULOS DE MOTOR Y MOTOCICLETAS', 'total': 89, 'distrito': 'ARGANZUELA          ', 'barrio': 'ATOCHA              '}
{'actividad_mas_frecuente': 'COMERCIO AL POR MAYOR Y AL POR MENOR; REPARACIÓN DE VEHÍCULOS DE MOTOR Y MOTOCICLETAS', 'total': 42, 'distrito': 'MORATALAZ           ', 'barrio': 'HORCAJO             '}
{'actividad_mas_frecuente': 'VALOR NULO EN ORIGEN', 'total': 632, 'distrito': 'HORTALEZA           ', 'barrio': 'CANILLAS            '}
{'actividad_mas_frecuente': 'COMERCIO AL POR MAYOR Y AL POR MENOR; REPARACIÓN DE VEHÍCULOS DE MOTOR Y MOTOCICLETAS', 'total': 594, 'distrito': 'CHAMBERI            ', 'barrio': 'TRAFALGAR           '}
{'actividad_mas_frecuente': 'VALOR NULO EN ORIGEN', 'total': 496, 'distrito': 'VILLAVERDE          ', 'barrio': 'LOS ROSALES         '}
{'actividad_mas_frecuente': 'COMERCIO AL POR MAYOR Y AL POR MENOR; REPARACIÓN DE VEHÍCULOS DE MOTOR Y MOTOCICLETAS', 'total': 1

f. Actualización de horarios de apertura y cierre de ciertos locales: modifica los
horarios de apertura y cierre de un conjunto seleccionado de locales según un
criterio que tú determines.

• Pista: primero, define el criterio que utilizarás para seleccionar los
locales a modificar (por ejemplo, su ubicación o actividad económica).
Usa $updateMany o $updateOne en combinación con $set para realizar
los cambios. Documenta el criterio elegido en tu informe y explica por
qué es relevante.


In [9]:
## condicion arbitraria, se modifican la fecha del distrito "ARGANZUA"

filtro = {
    "local.desc_distrito_local": {
                "$regex": "ARGANZUELA",
                "$options": "i"
            }
}

cantidad = collection.count_documents(filtro)

print("Documentos que se modificarían:", cantidad)


Documentos que se modificarían: 5933


In [13]:
update = {
    "$set": {
        "local.horario_apertura": "08:00",
        "local.horario_cierre": "22:00"
    }
}

resultado_f = collection.update_many(filtro, update)

print("Documentos modificados:", resultado_f.modified_count)


Documentos modificados: 5933


In [ ]:

import time
import json


def evaluar_pipeline(nombre, collection, pipeline, limite_preview=5):
    inicio = time.time()

    resultados = list(collection.aggregate(pipeline))

    fin = time.time()
    duracion = fin - inicio

    total = len(resultados)
    preview = resultados[:limite_preview]

    # Convertir pipeline y resultados a texto JSON bonito
    pipeline_str = json.dumps(pipeline, indent=2, ensure_ascii=False)
    preview_str = json.dumps(preview, indent=2, ensure_ascii=False)

    markdown = f"""
# Consulta {nombre}

## Pipeline utilizado

```json
{pipeline_str}
```

## Métricas

- Tiempo de ejecución: **{duracion:.4f} segundos**
- Total de documentos devueltos: **{total}**

## Ejemplo de resultados

```json
{preview_str}
```

---
"""

    return markdown, resultados


def generar_informe(collection, pipelines_dict, archivo_salida="informe_consultas.md"):
    informe = ""

    for nombre, pipeline in pipelines_dict.items():
        md, _ = evaluar_pipeline(nombre, collection, pipeline)
        informe += md

    with open(archivo_salida, "w", encoding="utf-8") as f:
        f.write(informe)

    print(f"Informe generado: {archivo_salida}")



In [12]:

pipelines = {
    "A": pipeline_a,
    "B": pipeline_b,
    "C": pipeline_c,
    "D": pipeline_d,
    "E": pipeline_e

}

generar_informe(collection, pipelines)

Informe generado: informe_consultas.md
